# 01 data audit

Audit of the PhysioNet/CinC Challenge 2019 sepsis training data (hospitals A and B) before any modelling.

## Setup

Imports, project root, paths, column groups and the logging helper.

In [1]:
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm import tqdm

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.3f}".format)


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "dataset").is_dir():
            return folder
    raise FileNotFoundError(f"no parent folder with a 'dataset' directory found from {start}")


ROOT = find_project_root()
DATASET_DIR = ROOT / "dataset"
SET_A_DIR = DATASET_DIR / "training_setA"
SET_B_DIR = DATASET_DIR / "training_setB"
KAGGLE_CSV = DATASET_DIR / "Dataset.csv"
PROCESSED_DIR = ROOT / "data" / "processed"
TABLES_DIR = ROOT / "results" / "tables"
FIGURES_DIR = ROOT / "results" / "figures"
for folder in (PROCESSED_DIR, TABLES_DIR, FIGURES_DIR):
    folder.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = PROCESSED_DIR / "sepsis_master.parquet"
GROUPS_PATH = PROCESSED_DIR / "column_groups.json"
ID_MAP_PATH = PROCESSED_DIR / "patient_id_map.csv"
SUMMARY_PATH = ROOT / "results" / "step01_summary.txt"

VITALS = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2"]
LABS = [
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2", "SaO2", "AST", "BUN", "Alkalinephos",
    "Calcium", "Chloride", "Creatinine", "Bilirubin_direct", "Glucose", "Lactate",
    "Magnesium", "Phosphate", "Potassium", "Bilirubin_total", "TroponinI", "Hct", "Hgb",
    "PTT", "WBC", "Fibrinogen", "Platelets",
]
STATIC = ["Age", "Gender", "Unit1", "Unit2", "HospAdmTime", "ICULOS"]
TARGET = "SepsisLabel"
FEATURES = VITALS + LABS + STATIC
ID_COLUMNS = ["patient_id", "hospital", "hour"]
HOSPITALS = ["A", "B"]

GROUP_OF = {c: "vital" for c in VITALS}
GROUP_OF.update({c: "lab" for c in LABS})
GROUP_OF.update({c: "static" for c in STATIC})
GROUP_OF[TARGET] = "target"

SUMMARY = []
SAVED_FILES = []


def log(text=""):
    print(text)
    SUMMARY.append(str(text))


log(f"project root folder: {ROOT.name}")

project root folder: Sepsis_ml


## Load the official files

Reading every .psv file from training_setA and training_setB in sorted order, with the file stem as patient id and the row position as hour.

In [2]:
start_time = time.perf_counter()
frames = []
files_per_hospital = {}
reference_columns = None
files_with_other_columns = []

for hospital, folder in [("A", SET_A_DIR), ("B", SET_B_DIR)]:
    files = sorted(folder.rglob("*.psv"))
    if not files:
        found = sorted(p.name for p in DATASET_DIR.iterdir())
        raise FileNotFoundError(f"no .psv files under {folder} (folder exists: {folder.exists()}); dataset folder contains: {found}")
    files_per_hospital[hospital] = len(files)
    for path in tqdm(files, desc=f"hospital {hospital}"):
        patient_frame = pd.read_csv(path, sep="|")
        if reference_columns is None:
            reference_columns = list(patient_frame.columns)
        elif list(patient_frame.columns) != reference_columns:
            files_with_other_columns.append(path.name)
        patient_frame.insert(0, "hour", np.arange(len(patient_frame), dtype="int32"))
        patient_frame.insert(0, "hospital", hospital)
        patient_frame.insert(0, "patient_id", path.stem)
        frames.append(patient_frame)

master = pd.concat(frames, ignore_index=True)
del frames
gc.collect()
FILE_COLUMNS = reference_columns
master = master[ID_COLUMNS + FILE_COLUMNS]
load_seconds = time.perf_counter() - start_time

n_files = sum(files_per_hospital.values())
log(f"files per hospital: A {files_per_hospital['A']:,}, B {files_per_hospital['B']:,}, total {n_files:,}")
log(f"load time: {load_seconds:.1f} s")
log(f"master shape: {master.shape[0]:,} rows x {master.shape[1]} columns")
log(f"patient ids unique across A and B: {master['patient_id'].nunique() == n_files} ({master['patient_id'].nunique():,} ids for {n_files:,} files)")
log(f"files whose columns differ from the first file: {len(files_with_other_columns)}")
display(master.head(3))

hospital B: 100%|██████████| 20000/20000 [00:33<00:00, 601.82it/s]


files per hospital: A 20,336, B 20,000, total 40,336
load time: 78.4 s
master shape: 1,552,210 rows x 44 columns
patient ids unique across A and B: True (40,336 ids for 40,336 files)
files whose columns differ from the first file: 0


,patient_id,hospital,hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,Bilirubin_total,TroponinI,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,p000001,A,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.140,0,NaN,NaN,-0.030,1,0
1,p000001,A,1,97.000,95.000,NaN,98.000,75.330,NaN,19.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.140,0,NaN,NaN,-0.030,2,0
2,p000001,A,2,89.000,99.000,NaN,122.000,86.000,NaN,22.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.140,0,NaN,NaN,-0.030,3,0


## Inspect Dataset.csv

Shape, columns and dtypes of the Kaggle file, and which columns look like a patient id and an hour index.

In [3]:
kaggle = pd.read_csv(KAGGLE_CSV)
log(f"Dataset.csv shape as read: {kaggle.shape[0]:,} rows x {kaggle.shape[1]} columns")

unnamed_columns = [c for c in kaggle.columns if str(c).lower().startswith("unnamed")]
kaggle = kaggle.drop(columns=unnamed_columns)
log(f"unnamed index columns dropped: {unnamed_columns or 'none'}")
log(f"Dataset.csv shape after drop: {kaggle.shape[0]:,} rows x {kaggle.shape[1]} columns")
log("Dataset.csv columns: " + ", ".join(map(str, kaggle.columns)))

dtype_counts = kaggle.dtypes.astype(str).value_counts()
log("Dataset.csv dtype counts: " + ", ".join(f"{k}: {v}" for k, v in dtype_counts.items()))
not_float = [c for c in kaggle.columns if str(kaggle[c].dtype) != "float64"]
log(f"Dataset.csv columns that are not float64: {not_float or 'none'}")
display(kaggle.head(3))

ID_NAMES = {"patient_id", "patientid", "patient", "pid", "id", "subject_id"}
HOUR_NAMES = {"hour", "hours", "time", "timestep", "t"}
kaggle_id_col = next((c for c in kaggle.columns if str(c).lower() in ID_NAMES), None)
kaggle_hour_col = next((c for c in kaggle.columns if str(c).lower() in HOUR_NAMES), None)
log(f"patient id column in Dataset.csv: {kaggle_id_col}")
log(f"hour index column in Dataset.csv: {kaggle_hour_col}")
if kaggle_id_col is not None:
    log(f"distinct patient ids in Dataset.csv: {kaggle[kaggle_id_col].nunique():,}")

Dataset.csv shape as read: 1,552,210 rows x 44 columns
unnamed index columns dropped: ['Unnamed: 0']
Dataset.csv shape after drop: 1,552,210 rows x 43 columns
Dataset.csv columns: Hour, HR, O2Sat, Temp, SBP, MAP, DBP, Resp, EtCO2, BaseExcess, HCO3, FiO2, pH, PaCO2, SaO2, AST, BUN, Alkalinephos, Calcium, Chloride, Creatinine, Bilirubin_direct, Glucose, Lactate, Magnesium, Phosphate, Potassium, Bilirubin_total, TroponinI, Hct, Hgb, PTT, WBC, Fibrinogen, Platelets, Age, Gender, Unit1, Unit2, HospAdmTime, ICULOS, SepsisLabel, Patient_ID
Dataset.csv dtype counts: float64: 38, int64: 5
Dataset.csv columns that are not float64: ['Hour', 'Gender', 'ICULOS', 'SepsisLabel', 'Patient_ID']


,Hour,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,FiO2,pH,PaCO2,SaO2,AST,BUN,Alkalinephos,Calcium,Chloride,Creatinine,Bilirubin_direct,Glucose,Lactate,Magnesium,Phosphate,Potassium,Bilirubin_total,TroponinI,Hct,Hgb,PTT,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel,Patient_ID
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.540,0,NaN,NaN,-0.020,1,0,17072
1,1,65.000,100.000,NaN,NaN,72.000,NaN,16.500,NaN,NaN,NaN,0.400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.540,0,NaN,NaN,-0.020,2,0,17072
2,2,78.000,100.000,NaN,NaN,42.500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68.540,0,NaN,NaN,-0.020,3,0,17072


patient id column in Dataset.csv: Patient_ID
hour index column in Dataset.csv: Hour
distinct patient ids in Dataset.csv: 40,336


## Compare Dataset.csv with the official files

First shape, columns and label total, then a row-for-row test in file order, and if that fails an alignment by patient id and ICULOS.

In [4]:
kaggle_features = [c for c in kaggle.columns if c not in (kaggle_id_col, kaggle_hour_col)]
same_rows = len(kaggle) == len(master)
missing_in_kaggle = [c for c in FILE_COLUMNS if c not in kaggle.columns]
extra_in_kaggle = [c for c in kaggle_features if c not in FILE_COLUMNS]

log(f"row count: official {len(master):,}, Dataset.csv {len(kaggle):,}, same: {same_rows}")
log(f"official columns missing in Dataset.csv: {missing_in_kaggle or 'none'}")
log(f"extra columns in Dataset.csv (besides id and hour): {extra_in_kaggle or 'none'}")
if TARGET in kaggle.columns:
    official_total, kaggle_total = int(master[TARGET].sum()), int(kaggle[TARGET].sum())
    log(f"{TARGET} total: official {official_total:,}, Dataset.csv {kaggle_total:,}, same: {official_total == kaggle_total}")


def mismatching_columns(columns, kaggle_order=None):
    mismatched = []
    for col in columns:
        official_values = master[col].to_numpy(dtype="float64")
        kaggle_values = kaggle[col].to_numpy(dtype="float64")
        if kaggle_order is not None:
            kaggle_values = kaggle_values[kaggle_order]
        if not np.allclose(official_values, kaggle_values, equal_nan=True):
            mismatched.append(col)
    return mismatched


match_mode = None
kaggle_order = None
try:
    if same_rows and not missing_in_kaggle:
        mismatched = mismatching_columns(FILE_COLUMNS)
        log(f"file order test: {len(mismatched)} of {len(FILE_COLUMNS)} columns differ {mismatched or ''}".rstrip())
        if not mismatched:
            match_mode = "file order"
    else:
        log("file order test skipped because row count or columns differ")
except Exception as error:
    log(f"file order test failed with {type(error).__name__}: {error}")

row count: official 1,552,210, Dataset.csv 1,552,210, same: True
official columns missing in Dataset.csv: none
extra columns in Dataset.csv (besides id and hour): none
SepsisLabel total: official 27,916, Dataset.csv 27,916, same: True
file order test: 41 of 41 columns differ ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2', 'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN', 'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct', 'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium', 'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC', 'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2', 'HospAdmTime', 'ICULOS', 'SepsisLabel']


In [5]:
alignment_note = "not attempted"
if match_mode is None and kaggle_id_col is not None and "ICULOS" in kaggle.columns:
    try:
        kaggle_ids = kaggle[kaggle_id_col]
        official_ids = master["patient_id"]
        if kaggle_ids.astype(str).isin(set(official_ids.unique())).all():
            aligned_ids = kaggle_ids.astype(str).to_numpy()
            alignment_note = "ids already in official format"
        elif kaggle_ids.nunique() == official_ids.nunique():
            kaggle_sorted = np.sort(kaggle_ids.unique())
            official_sorted = np.sort(official_ids.unique())
            aligned_ids = kaggle_ids.map(dict(zip(kaggle_sorted, official_sorted))).to_numpy()
            alignment_note = "ids matched by sorted rank"
        else:
            aligned_ids = None
            alignment_note = "patient counts differ, no alignment possible"

        if aligned_ids is not None:
            kaggle_index = pd.MultiIndex.from_arrays([aligned_ids, kaggle["ICULOS"].to_numpy()])
            official_index = pd.MultiIndex.from_arrays([official_ids.to_numpy(), master["ICULOS"].to_numpy()])
            if len(kaggle_index) == len(official_index) and kaggle_index.is_unique and official_index.is_unique:
                positions = official_index.get_indexer(kaggle_index)
                if (positions >= 0).all():
                    kaggle_order = np.argsort(positions)
                    mismatched = mismatching_columns(FILE_COLUMNS, kaggle_order)
                    log(f"alignment by patient id + ICULOS ({alignment_note}): {len(mismatched)} of {len(FILE_COLUMNS)} columns differ {mismatched or ''}".rstrip())
                    if not mismatched:
                        match_mode = "patient id + ICULOS"
                else:
                    alignment_note += "; some Dataset.csv keys not found in official files"
            else:
                alignment_note += "; keys not unique or row counts differ"
    except Exception as error:
        alignment_note = f"failed with {type(error).__name__}: {error}"
    if match_mode is None:
        log(f"alignment by patient id + ICULOS did not produce a match ({alignment_note})")
elif match_mode is None:
    log("alignment skipped: Dataset.csv has no patient id column or no ICULOS column")

alignment by patient id + ICULOS (ids matched by sorted rank): 0 of 41 columns differ


In [6]:
mapping_saved = False
if match_mode is not None and kaggle_id_col is not None:
    kaggle_ids_in_master_order = kaggle[kaggle_id_col].to_numpy()
    if kaggle_order is not None:
        kaggle_ids_in_master_order = kaggle_ids_in_master_order[kaggle_order]
    id_map = pd.DataFrame({
        "dataset_csv_patient_id": kaggle_ids_in_master_order,
        "patient_id": master["patient_id"].to_numpy(),
        "hospital": master["hospital"].to_numpy(),
    }).drop_duplicates().reset_index(drop=True)
    one_to_one = len(id_map) == id_map["patient_id"].nunique() == id_map["dataset_csv_patient_id"].nunique()
    id_map.to_csv(ID_MAP_PATH, index=False)
    SAVED_FILES.append(ID_MAP_PATH)
    mapping_saved = True
    log(f"id map rows: {len(id_map):,}, one-to-one: {one_to_one}")
    display(id_map.head(3))
elif match_mode is not None:
    log("data match but Dataset.csv has no patient id column, so no id map was built")
else:
    log("id map skipped because the two sources do not match")

if match_mode is not None:
    log(f"conclusion: Dataset.csv holds the same data as the official A/B files ({match_mode}); the official files are the master because they carry the hospital label.")
else:
    log("conclusion: Dataset.csv does not match the official A/B files row for row; the official files are the master because they carry the hospital label.")

del kaggle
gc.collect();

id map rows: 40,336, one-to-one: True


,dataset_csv_patient_id,patient_id,hospital
0,1,p000001,A
1,2,p000002,A
2,3,p000003,A


conclusion: Dataset.csv holds the same data as the official A/B files (patient id + ICULOS); the official files are the master because they carry the hospital label.


## Core audit

Counts, stay lengths, missingness, value ranges, sanity checks and onset information on the master table. Nothing is modified.

In [7]:
expected_columns = FEATURES + [TARGET]
missing_columns = [c for c in expected_columns if c not in master.columns]
extra_columns = [c for c in FILE_COLUMNS if c not in expected_columns]
log(f"expected columns missing from master: {missing_columns or 'none'}")
log(f"extra columns in master: {extra_columns or 'none'}")
log(f"column order matches the expected groups: {FILE_COLUMNS == expected_columns}")
log(f"feature columns: {len(FILE_COLUMNS) - 1}, target: {TARGET}")

grouped = master.groupby("patient_id", sort=False)
patients = pd.DataFrame({
    "hospital": grouped["hospital"].first(),
    "n_rows": grouped.size(),
    "septic": grouped[TARGET].max() == 1,
})
rows_by_hospital = master.groupby("hospital")[TARGET].agg(rows="size", positive_rows="sum")

n_patients, n_rows = len(patients), len(master)
log(f"patients: {n_patients:,} (expected about 40,336)")
log(f"rows: {n_rows:,} (expected about 1.5 million)")
for h in HOSPITALS:
    log(f"hospital {h}: {int((patients['hospital'] == h).sum()):,} patients, {int(rows_by_hospital.loc[h, 'rows']):,} rows")

n_septic = int(patients["septic"].sum())
n_positive = int(master[TARGET].sum())
log(f"septic patients: {n_septic:,} ({100 * n_septic / n_patients:.2f}%, expected about 7%)")
log(f"positive rows: {n_positive:,} ({100 * n_positive / n_rows:.2f}%, expected about 2%)")
for h in HOSPITALS:
    sub = patients[patients["hospital"] == h]
    h_rows = int(rows_by_hospital.loc[h, "rows"])
    h_pos = int(rows_by_hospital.loc[h, "positive_rows"])
    log(f"hospital {h}: septic patients {int(sub['septic'].sum()):,} ({100 * sub['septic'].mean():.2f}%), positive rows {h_pos:,} ({100 * h_pos / h_rows:.2f}%)")


def rows_summary(series):
    low, q1, median, q3, high = series.quantile([0, 0.25, 0.5, 0.75, 1]).to_numpy()
    return f"min {low:.0f}, 25% {q1:.0f}, median {median:.0f}, 75% {q3:.0f}, max {high:.0f}"


log(f"rows per patient overall: {rows_summary(patients['n_rows'])}")
for h in HOSPITALS:
    log(f"rows per patient hospital {h}: {rows_summary(patients.loc[patients['hospital'] == h, 'n_rows'])}")

stay_median = patients.groupby("septic")["n_rows"].median()
log(f"median stay (hours): septic {stay_median[True]:.1f}, non-septic {stay_median[False]:.1f}")

expected columns missing from master: none
extra columns in master: none
column order matches the expected groups: True
feature columns: 40, target: SepsisLabel
patients: 40,336 (expected about 40,336)
rows: 1,552,210 (expected about 1.5 million)
hospital A: 20,336 patients, 790,215 rows
hospital B: 20,000 patients, 761,995 rows
septic patients: 2,932 (7.27%, expected about 7%)
positive rows: 27,916 (1.80%, expected about 2%)
hospital A: septic patients 1,790 (8.80%), positive rows 17,136 (2.17%)
hospital B: septic patients 1,142 (5.71%), positive rows 10,780 (1.41%)
rows per patient overall: min 8, 25% 24, median 38, 75% 47, max 336
rows per patient hospital A: min 8, 25% 25, median 39, 75% 47, max 336
rows per patient hospital B: min 8, 25% 23, median 38, 75% 47, max 336
median stay (hours): septic 37.5, non-septic 39.0


In [8]:
missing_counts = master[FILE_COLUMNS].isna().sum()
missingness = pd.DataFrame({
    "column": missing_counts.index,
    "group": [GROUP_OF.get(c, "extra") for c in missing_counts.index],
    "missing_count": missing_counts.to_numpy(),
    "missing_pct": 100 * missing_counts.to_numpy() / len(master),
}).sort_values("missing_pct", ascending=False).reset_index(drop=True)
display(missingness)

group_missing = missingness.groupby("group")["missing_pct"].mean().sort_values(ascending=False)
log("mean missing % per group: " + ", ".join(f"{g} {v:.1f}" for g, v in group_missing.items()))
log("highest missing %: " + ", ".join(f"{r.column} {r.missing_pct:.1f}" for r in missingness.head(5).itertuples()))
log("lowest missing %: " + ", ".join(f"{r.column} {r.missing_pct:.1f}" for r in missingness.tail(5).itertuples()))
log(f"columns with more than 90% missing: {int((missingness['missing_pct'] > 90).sum())}")

,column,group,missing_count,missing_pct
0,Bilirubin_direct,lab,1549220,99.807
1,Fibrinogen,lab,1541968,99.340
2,TroponinI,lab,1537429,99.048
3,Bilirubin_total,lab,1529069,98.509
4,Alkalinephos,lab,1527269,98.393
5,AST,lab,1527027,98.378
6,Lactate,lab,1510764,97.330
7,PTT,lab,1506511,97.056
8,SaO2,lab,1498649,96.549
9,EtCO2,vital,1494574,96.287


mean missing % per group: lab 94.8, vital 32.4, static 13.1, target 0.0
highest missing %: Bilirubin_direct 99.8, Fibrinogen 99.3, TroponinI 99.0, Bilirubin_total 98.5, Alkalinephos 98.4
lowest missing %: HospAdmTime 0.0, Gender 0.0, Age 0.0, ICULOS 0.0, SepsisLabel 0.0
columns with more than 90% missing: 26


In [9]:
dtype_summary = master[FILE_COLUMNS].dtypes.astype(str).value_counts()
log("master dtype counts: " + ", ".join(f"{k}: {v}" for k, v in dtype_summary.items()))

describe_table = master[FILE_COLUMNS].describe().T.rename(columns={"50%": "median"})
describe_table = describe_table[["count", "min", "max", "mean", "median", "std"]]
describe_table.insert(0, "group", [GROUP_OF.get(c, "extra") for c in describe_table.index])
describe_table.insert(1, "dtype", master[FILE_COLUMNS].dtypes.astype(str).to_numpy())

PLAUSIBLE_RANGES = {
    "HR": (20, 250), "O2Sat": (50, 100), "Temp": (30, 43), "SBP": (40, 300),
    "MAP": (20, 200), "DBP": (10, 200), "Resp": (4, 70), "Age": (0, 100),
    "pH": (6.5, 7.8), "FiO2": (0, 1),
}
range_notes = {}
for col, (low, high) in PLAUSIBLE_RANGES.items():
    values = master[col]
    n_outside = int(((values < low) | (values > high)).sum())
    observed = f"observed {values.min():.2f} to {values.max():.2f}"
    if n_outside:
        range_notes[col] = f"{n_outside:,} values outside {low} to {high}"
        log(f"suspicious range, {col}: {range_notes[col]} ({observed})")
    else:
        log(f"range looks plausible, {col}: {observed}")
describe_table["note"] = [range_notes.get(c, "") for c in describe_table.index]
display(describe_table.round(3))

master dtype counts: float64: 38, int64: 3
suspicious range, HR: 4 values outside 20 to 250 (observed 20.00 to 280.00)
suspicious range, O2Sat: 272 values outside 50 to 100 (observed 20.00 to 100.00)
suspicious range, Temp: 18 values outside 30 to 43 (observed 20.90 to 50.00)
suspicious range, SBP: 114 values outside 40 to 300 (observed 20.00 to 300.00)
suspicious range, MAP: 444 values outside 20 to 200 (observed 20.00 to 300.00)
suspicious range, DBP: 103 values outside 10 to 200 (observed 20.00 to 300.00)
suspicious range, Resp: 1,851 values outside 4 to 70 (observed 1.00 to 100.00)
range looks plausible, Age: observed 14.00 to 100.00
suspicious range, pH: 1 values outside 6.5 to 7.8 (observed 6.62 to 7.93)
suspicious range, FiO2: 32 values outside 0 to 1 (observed -50.00 to 4000.00)


,group,dtype,count,min,max,mean,median,std,note
HR,vital,float64,"1,398,811.000",20.000,280.000,84.581,83.500,17.325,4 values outside 20 to 250
O2Sat,vital,float64,"1,349,474.000",20.000,100.000,97.194,98.000,2.937,272 values outside 50 to 100
Temp,vital,float64,"525,226.000",20.900,50.000,36.977,37.000,0.770,18 values outside 30 to 43
SBP,vital,float64,"1,325,945.000",20.000,300.000,123.750,121.000,23.232,114 values outside 40 to 300
MAP,vital,float64,"1,358,940.000",20.000,300.000,82.400,80.000,16.342,444 values outside 20 to 200
DBP,vital,float64,"1,065,656.000",20.000,300.000,63.831,62.000,13.956,103 values outside 10 to 200
Resp,vital,float64,"1,313,875.000",1.000,100.000,18.726,18.000,5.098,"1,851 values outside 4 to 70"
EtCO2,vital,float64,"57,636.000",10.000,100.000,32.958,33.000,7.952,
BaseExcess,lab,float64,"84,145.000",-32.000,100.000,-0.690,0.000,4.294,
HCO3,lab,float64,"65,028.000",0.000,55.000,24.075,24.000,4.377,


In [10]:
checks = []


def add_check(name, violations, detail="", status=None):
    status = status or ("pass" if violations == 0 else "fail")
    checks.append({"check": name, "violations": int(violations), "status": status, "detail": detail})
    log(f"[{status}] {name}: {int(violations):,} violations {detail}".rstrip())


patient_codes = pd.factorize(master["patient_id"])[0]
same_patient = patient_codes[1:] == patient_codes[:-1]

iculos_steps = np.diff(master["ICULOS"].to_numpy(dtype="float64"))
add_check("ICULOS strictly increasing within patient", (same_patient & (iculos_steps <= 0)).sum())
add_check("ICULOS has no missing values", master["ICULOS"].isna().sum())

label_steps = np.diff(master[TARGET].to_numpy(dtype="int64"))
add_check("label never goes from 1 back to 0", (same_patient & (label_steps < 0)).sum())

duplicates = pd.DataFrame({"p": patient_codes, "h": master["hour"].to_numpy()}).duplicated().sum()
add_check("no duplicated (patient_id, hour) pairs", duplicates)

add_check("Unit1 and Unit2 never both 1", ((master["Unit1"] == 1) & (master["Unit2"] == 1)).sum())
both_unit_missing = int((master["Unit1"].isna() & master["Unit2"].isna()).sum())
add_check("Unit1 and Unit2 both missing (rows)", both_unit_missing,
          f"({100 * both_unit_missing / len(master):.2f}% of rows)", status="info")

gender_counts = master["Gender"].value_counts(dropna=False)
log("Gender value counts: " + ", ".join(f"{k}: {v:,}" for k, v in gender_counts.items()))
add_check("Gender only 0 or 1 (or missing)", (master["Gender"].notna() & ~master["Gender"].isin([0, 1])).sum())

age = master["Age"]
add_check("Age within 0 to 100", ((age < 0) | (age > 100)).sum(), f"(min {age.min():.1f}, max {age.max():.1f})")

hosp_adm = master["HospAdmTime"]
add_check("HospAdmTime not positive", (hosp_adm > 0).sum(), f"(min {hosp_adm.min():.2f}, max {hosp_adm.max():.2f})")

sanity_checks = pd.DataFrame(checks)
display(sanity_checks)

[pass] ICULOS strictly increasing within patient: 0 violations
[pass] ICULOS has no missing values: 0 violations
[pass] label never goes from 1 back to 0: 0 violations
[pass] no duplicated (patient_id, hour) pairs: 0 violations
[pass] Unit1 and Unit2 never both 1: 0 violations
[info] Unit1 and Unit2 both missing (rows): 611,960 violations (39.43% of rows)
Gender value counts: 1: 868,103, 0: 684,107
[pass] Gender only 0 or 1 (or missing): 0 violations
[pass] Age within 0 to 100: 0 violations (min 14.0, max 100.0)
[fail] HospAdmTime not positive: 8,513 violations (min -5366.86, max 23.99)


,check,violations,status,detail
0,ICULOS strictly increasing within patient,0,pass,
1,ICULOS has no missing values,0,pass,
2,label never goes from 1 back to 0,0,pass,
3,"no duplicated (patient_id, hour) pairs",0,pass,
4,Unit1 and Unit2 never both 1,0,pass,
5,Unit1 and Unit2 both missing (rows),611960,info,(39.43% of rows)
6,Gender only 0 or 1 (or missing),0,pass,
7,Age within 0 to 100,0,pass,"(min 14.0, max 100.0)"
8,HospAdmTime not positive,8513,fail,"(min -5366.86, max 23.99)"


In [11]:
first_positive_hour = master.loc[master[TARGET] == 1].groupby("patient_id")["hour"].min()
log(f"septic patients with a positive row: {len(first_positive_hour):,} (septic patients counted earlier: {n_septic:,})")
log(f"first positive hour: min {first_positive_hour.min()}, median {first_positive_hour.median():.1f}, max {first_positive_hour.max()}")
at_start = int((first_positive_hour == 0).sum())
log(f"septic patients already labelled 1 in their first row: {at_start:,} ({100 * at_start / len(first_positive_hour):.2f}%)")

short_stay = patients["n_rows"] < 24
for is_septic, name in [(False, "non-septic"), (True, "septic")]:
    in_group = patients["septic"] == is_septic
    n_short = int((short_stay & in_group).sum())
    log(f"{name} patients with fewer than 24 rows: {n_short:,} of {int(in_group.sum()):,} ({100 * n_short / in_group.sum():.2f}%)")

septic patients with a positive row: 2,932 (septic patients counted earlier: 2,932)
first positive hour: min 0, median 28.0, max 330
septic patients already labelled 1 in their first row: 426 (14.53%)
non-septic patients with fewer than 24 rows: 8,587 of 37,404 (22.96%)
septic patients with fewer than 24 rows: 1,088 of 2,932 (37.11%)


## Tables

Table 1 and the audit tables saved to results/tables.

In [12]:
def summary_row(pat, n_rows_total, n_positive_total):
    return {
        "patients": len(pat),
        "rows": int(n_rows_total),
        "septic patients": int(pat["septic"].sum()),
        "septic patients %": 100 * pat["septic"].mean(),
        "positive rows": int(n_positive_total),
        "positive rows %": 100 * n_positive_total / n_rows_total,
        "median stay (hours)": pat["n_rows"].median(),
        "mean stay (hours)": pat["n_rows"].mean(),
    }


table1_rows = [summary_row(patients, n_rows, n_positive)]
for h in HOSPITALS:
    table1_rows.append(summary_row(
        patients[patients["hospital"] == h],
        rows_by_hospital.loc[h, "rows"],
        rows_by_hospital.loc[h, "positive_rows"],
    ))
table1 = pd.DataFrame(table1_rows, index=["Overall", "Hospital A", "Hospital B"])
display(table1.round(3))

table_files = {
    "table1_dataset_summary.csv": (table1.round(3), True),
    "missingness_by_column.csv": (missingness.round(3), False),
    "column_describe.csv": (describe_table.round(3), True),
    "sanity_checks.csv": (sanity_checks, False),
}
for file_name, (table, keep_index) in table_files.items():
    path = TABLES_DIR / file_name
    table.to_csv(path, index=keep_index, index_label="name" if keep_index else None)
    SAVED_FILES.append(path)
    log(f"saved {path.relative_to(ROOT)}")

,patients,rows,septic patients,septic patients %,positive rows,positive rows %,median stay (hours),mean stay (hours)
Overall,40336,1552210,2932,7.269,27916,1.798,38.000,38.482
Hospital A,20336,790215,1790,8.802,17136,2.169,39.000,38.858
Hospital B,20000,761995,1142,5.710,10780,1.415,38.000,38.100


saved results/tables/table1_dataset_summary.csv
saved results/tables/missingness_by_column.csv
saved results/tables/column_describe.csv
saved results/tables/sanity_checks.csv


## Save the master table

Parquet file with compact dtypes, the column groups as json, and a re-read check.

In [13]:
dtype_map = {c: "float32" for c in FEATURES}
dtype_map.update({TARGET: "int8", "hour": "int16"})
master_out = master.astype(dtype_map)
master_out = master_out.sort_values(["hospital", "patient_id", "hour"], kind="stable").reset_index(drop=True)
master_out.to_parquet(PARQUET_PATH, index=False)
SAVED_FILES.append(PARQUET_PATH)

column_groups = {
    "vitals": VITALS,
    "labs": LABS,
    "static": STATIC,
    "target": [TARGET],
    "id_columns": ID_COLUMNS,
}
with open(GROUPS_PATH, "w") as f:
    json.dump(column_groups, f, indent=2)
SAVED_FILES.append(GROUPS_PATH)

reread = pd.read_parquet(PARQUET_PATH)
shape_ok = reread.shape == master_out.shape
dtypes_ok = reread.dtypes.astype(str).equals(master_out.dtypes.astype(str))
log(f"parquet re-read shape {reread.shape}, matches written table: {shape_ok}")
log(f"parquet re-read dtypes match written table: {dtypes_ok}")
log("saved dtypes: " + ", ".join(f"{k}: {v}" for k, v in reread.dtypes.astype(str).value_counts().items()))
log(f"sorted by hospital, patient_id, hour: {master_out[['hospital', 'patient_id', 'hour']].equals(reread[['hospital', 'patient_id', 'hour']])}")
del master_out, reread
gc.collect();

parquet re-read shape (1552210, 44), matches written table: True
parquet re-read dtypes match written table: True
saved dtypes: float32: 40, str: 2, int16: 1, int8: 1
sorted by hospital, patient_id, hour: True


## Saved files

Checklist of everything written, then the summary text file.

In [14]:
log("saved files:")
for path in SAVED_FILES:
    log(f"  {path.relative_to(ROOT)}  ({path.stat().st_size / 1e6:.3f} MB)")
log(f"  {SUMMARY_PATH.relative_to(ROOT)}  (written next)")

SUMMARY_PATH.write_text("\n".join(SUMMARY) + "\n")
print(SUMMARY_PATH.relative_to(ROOT))

saved files:
  data/processed/patient_id_map.csv  (0.654 MB)
  results/tables/table1_dataset_summary.csv  (0.000 MB)
  results/tables/missingness_by_column.csv  (0.001 MB)
  results/tables/column_describe.csv  (0.003 MB)
  results/tables/sanity_checks.csv  (0.000 MB)
  data/processed/sepsis_master.parquet  (17.182 MB)
  data/processed/column_groups.json  (0.001 MB)
  results/step01_summary.txt  (written next)
results/step01_summary.txt
